# Item-to-item retrieval: LLM retrieval hints

**Goal:** Measure whether **short LLM-generated `RETRIEVAL_HINT` lines** (grounded in table facts) improve the same **item-to-item co-preference** protocol as Notebooks 2/3.

**Prereq:** generate `dish_retrieval_aug.json` (local or anywhere), then make it available to this notebook.

```bash
python scripts/augment_dishes_retrieval_llm.py --dishes-parquet data/synthetic/dishes.parquet
```

This writes `data/dish_retrieval_aug.json` mapping `id -> retrieval_hint`.

**Kaggle:** add `dish_retrieval_aug.json` as a **Dataset input** (any folder name). The notebook searches `/kaggle/input/**/dish_retrieval_aug.json` automatically.

**Why this can move P@10 here:** synthetic users co-order within similar `cuisine_cluster` + meal-time patterns. Hints make those latent co-purchase axes explicit in the embedded text.


In [ ]:
import torch

assert torch.cuda.is_available(), "GPU recommended for encoding (enable GPU on Colab/Kaggle)."
print(f"✓ GPU: {torch.cuda.get_device_name(0)}")


In [ ]:
# ============================================================
# SETUP
# ============================================================
import os, sys

REPO = "Embedding-Based-Recommender"
GITHUB_USER = "IldarRakiev"

ENV = 'kaggle' if os.path.exists('/kaggle/working') else 'colab' if os.path.exists('/content') else 'local'
BASE = '/kaggle/working' if ENV == 'kaggle' else '/content' if ENV == 'colab' else os.getcwd()
REPO_DIR = f'{BASE}/{REPO}' if ENV != 'local' else os.path.abspath(os.path.join(os.getcwd(), '..'))

if ENV != 'local':
    if not os.path.exists(REPO_DIR):
        os.system(f'git clone https://github.com/{GITHUB_USER}/{REPO}.git {REPO_DIR}')
    else:
        os.system(f'cd {REPO_DIR} && git pull -q')

os.system('pip install -q sentence-transformers faiss-cpu pandas pyarrow tqdm')
sys.path.insert(0, f'{REPO_DIR}/src')

print(f"Environment: {ENV} | Repo: {REPO_DIR}")
print("Setup complete")


In [ ]:
# ============================================================
# DATA PATHS
# ============================================================
import os
import glob as _glob

ENV = 'kaggle' if os.path.exists('/kaggle/working') else 'colab' if os.path.exists('/content') else 'local'

if ENV == 'local':
    SYNTHETIC_DIR = os.path.join(os.path.dirname(os.getcwd()), 'data', 'synthetic')
else:
    SYNTHETIC_DIR = os.path.join(REPO_DIR, 'data', 'synthetic')

OUTPUT_DIR = '/kaggle/working/processed' if ENV == 'kaggle' else SYNTHETIC_DIR
os.makedirs(OUTPUT_DIR, exist_ok=True)

AUG_PATH_CANDIDATES = [
    os.path.join(REPO_DIR, 'data', 'dish_retrieval_aug.json'),
    os.path.join(os.path.dirname(SYNTHETIC_DIR), 'dish_retrieval_aug.json'),
    os.path.join(SYNTHETIC_DIR, 'dish_retrieval_aug.json'),
]

# Kaggle/Colab: allow uploading augmentation JSON as an Input dataset
if os.path.isdir('/kaggle/input'):
    AUG_PATH_CANDIDATES.extend(_glob.glob('/kaggle/input/**/dish_retrieval_aug.json', recursive=True))
if os.path.isdir('/content'):
    AUG_PATH_CANDIDATES.extend(_glob.glob('/content/**/dish_retrieval_aug.json', recursive=True))

print(f"SYNTHETIC_DIR = {SYNTHETIC_DIR}")
print(f"OUTPUT_DIR    = {OUTPUT_DIR}")


In [ ]:
import json
import faiss
import numpy as np
import pandas as pd

from text_builders import dish_to_rich_text
from embedding_model import EmbeddingModel
from utils import evaluate_all

np.random.seed(42)

dishes = pd.read_parquet(f"{SYNTHETIC_DIR}/dishes.parquet")
test = pd.read_parquet(f"{SYNTHETIC_DIR}/interactions_test.parquet")

dish_id_to_idx = {did: i for i, did in enumerate(dishes['id'])}
idx_to_dish_id = {i: did for i, did in enumerate(dishes['id'])}

aug_map = {}
aug_path = next((p for p in AUG_PATH_CANDIDATES if os.path.isfile(p)), None)
if not aug_path:
    raise FileNotFoundError(
        "dish_retrieval_aug.json not found. Run:\n"
        "  python scripts/augment_dishes_retrieval_llm.py\n"
        f"Tried: {AUG_PATH_CANDIDATES}"
    )

aug_rows = json.loads(open(aug_path, 'r', encoding='utf-8').read())
for row in aug_rows:
    aug_map[int(row['id'])] = str(row.get('retrieval_hint') or '').strip()

print(f"Loaded hints: {len(aug_map):,} from {aug_path}")

FLAG_KW = dict(
    include_recipe=False,
    include_macro_tokens=False,
    include_ratios=True,
    include_ingredients=True,
)

texts_base = []
texts_aug = []
for _, row in dishes.iterrows():
    rid = int(row['id'])
    base = dish_to_rich_text(row.to_dict(), tags=row.get('tag_list', []), **FLAG_KW)
    texts_base.append(base)
    texts_aug.append(
        dish_to_rich_text(
            row.to_dict(),
            tags=row.get('tag_list', []),
            retrieval_hint=aug_map.get(rid, ""),
            **FLAG_KW,
        )
    )

print("Built texts: baseline vs augmented")


In [ ]:
model = EmbeddingModel()
print(f"Model: {model.model_name} | dim={model.dim}")

embs_base = model.encode(texts_base, batch_size=64).astype(np.float32)
embs_aug = model.encode(texts_aug, batch_size=64).astype(np.float32)

idx_base = faiss.IndexFlatIP(model.dim)
idx_base.add(embs_base)

idx_aug = faiss.IndexFlatIP(model.dim)
idx_aug.add(embs_aug)

print("FAISS indexes ready")


In [ ]:
def evaluate_item_to_item(index, embeddings, ks=None):
    if ks is None:
        ks = [5, 10, 20]

    user_positives = (
        test[test['interaction_type'].isin(['order', 'favorite'])]
        .groupby('user_id')['dish_id']
        .apply(set)
        .to_dict()
    )

    rows = []
    for _, pos_dishes in user_positives.items():
        pos_dishes = {d for d in pos_dishes if d in dish_id_to_idx}
        if len(pos_dishes) < 5:
            continue

        query_dish = list(pos_dishes)[0]
        relevant = pos_dishes - {query_dish}

        q_idx = dish_id_to_idx[query_dish]
        _, inds = index.search(embeddings[q_idx:q_idx + 1], max(ks) + 1)
        recommended = [idx_to_dish_id[i] for i in inds[0] if i >= 0 and idx_to_dish_id.get(i) != query_dish]
        rows.append(evaluate_all(recommended, relevant, ks=ks))

    return pd.DataFrame(rows).mean().to_dict() if rows else {}


m_base = evaluate_item_to_item(idx_base, embs_base)
m_aug = evaluate_item_to_item(idx_aug, embs_aug)

cmp = pd.DataFrame({"baseline_text": m_base, "+llm_retrieval_hint": m_aug}).T
print(cmp[["P@5", "P@10", "NDCG@10", "MRR"]].round(4))

print(f"\nΔ P@10: {m_aug.get('P@10', 0) - m_base.get('P@10', 0):+.4f}")


In [ ]:
out = {"baseline_text": m_base, "llm_retrieval_hint": m_aug, "aug_path": aug_path}

with open(os.path.join(OUTPUT_DIR, "results_item_to_item_llm_retrieval_aug.json"), "w", encoding="utf-8") as f:
    json.dump(out, f, indent=2)

print(f"Saved: {os.path.join(OUTPUT_DIR, 'results_item_to_item_llm_retrieval_aug.json')}")
